#### ORCHESTRATOR

##### 11.1 DOCUMENTAÇÃO DO FRAMEWORK ORCHESTRATOR

###### Objetivo

O **Framework Orchestrator** é responsável por planejar, coordenar, executar e monitorar os diferentes componentes analíticos do Data Quality Framework, utilizando recursos do **Databricks** e do **PySpark**.

Seu objetivo é centralizar o controle operacional dos Profiles, garantindo que as execuções sejam organizadas conforme as dependências configuradas, que os resultados operacionais sejam rastreáveis e que eventuais falhas possam ser identificadas.

O Orchestrator não realiza diretamente as análises estatísticas ou estruturais dos dados. Essas responsabilidades permanecem nos respectivos Profiles.

A análise busca responder:

> Quais componentes do Data Quality Framework foram planejados e executados, quais apresentaram falhas e qual foi o desempenho operacional da execução?

O Orchestrator possui caráter **operacional e de coordenação**. O sucesso de sua execução não representa, necessariamente, a aprovação da qualidade do dataset analisado.

---

###### Exemplos

**Exemplo 1 — Execução de um único Profile**

Considere a seleção:

`ORCHESTRATOR_SELECTED = ["schema"]`

O Orchestrator identifica o Profile solicitado, verifica suas dependências e constrói o plano necessário para executá-lo.

O resultado esperado é um registro operacional contendo o identificador da execução, o Profile, seu status e a duração do processamento.

**Exemplo 2 — Execução de múltiplos Profiles**

Considere a seleção:

`ORCHESTRATOR_SELECTED = ["schema", "distribution", "pattern"]`

O Orchestrator constrói um plano contendo os Profiles selecionados e suas dependências, respeitando a ordem necessária para execução.

A sequência efetiva depende das relações definidas em `ORCHESTRATOR_CATALOG`.

**Exemplo 3 — Falha durante a execução**

Considere que determinado Profile apresente uma exceção durante o processamento.

O Orchestrator registra:

- O Profile que apresentou a falha;
- O status `ERRO`;
- O horário de início;
- O horário de término;
- A duração registrada;
- A mensagem associada à exceção.

Quando `ORCHESTRATOR_STOP_ON_ERROR` estiver habilitado, o motor interrompe o processamento após registrar a falha.

**Exemplo 4 — Dependência bloqueada**

Considere que um Profile dependa de outro componente que apresentou falha.

Quando a execução continuar após o erro, o Profile dependente poderá receber o status `BLOQUEADO`, indicando que não foi executado porque sua dependência não foi concluída corretamente.

**Exemplo 5 — Monitoramento do desempenho**

Considere uma execução que apresente diferentes durações entre os Profiles.

O Orchestrator registra os tempos individuais, permitindo identificar quais componentes demandaram maior tempo de processamento.

Essas informações poderão orientar futuras otimizações de performance.

---

###### Principais análises

O Framework Orchestrator contempla:

1. **Configuração da execução**
   - Definição dos Profiles disponíveis;
   - Seleção dos componentes a executar;
   - Configuração do tempo limite;
   - Definição do comportamento diante de falhas;
   - Definição de argumentos para notebooks filhos;
   - Identificação única da execução.

2. **Validação do catálogo**
   - Verificação dos Profiles cadastrados;
   - Validação dos caminhos dos notebooks;
   - Verificação das dependências declaradas;
   - Identificação de referências inválidas.

3. **Planejamento da execução**
   - Identificação dos Profiles selecionados;
   - Inclusão das dependências necessárias;
   - Ordenação dos componentes;
   - Prevenção de duplicações;
   - Detecção de dependências cíclicas.

4. **Execução dos notebooks**
   - Invocação dos notebooks selecionados;
   - Aplicação dos parâmetros configurados;
   - Controle de tempo limite;
   - Captura de exceções;
   - Interrupção configurável em caso de falha.

5. **Monitoramento operacional**
   - Registro de início e término;
   - Cálculo da duração;
   - Classificação do status;
   - Registro de mensagens de erro;
   - Associação dos resultados ao identificador da execução.

6. **Consolidação dos resultados**
   - Contagem de Profiles executados;
   - Identificação de falhas;
   - Identificação de bloqueios;
   - Agregação das durações;
   - Construção de indicadores executivos.

7. **Validação dos contratos**
   - Verificação do schema;
   - Validação dos status;
   - Verificação dos identificadores;
   - Validação da ordem das dependências;
   - Verificação da consistência do plano.

---

###### Métodos de execução

**Catálogo de Profiles**

O catálogo `ORCHESTRATOR_CATALOG` estabelece a relação entre os componentes disponíveis, seus caminhos e suas dependências.

Essa estrutura permite modificar a composição do plano sem alterar diretamente a lógica do motor.

**Planejamento por dependências**

A função `dq_orchestrator_plan()` organiza os Profiles selecionados considerando as dependências declaradas.

O planejamento busca assegurar que cada dependência apareça antes do componente que necessita dela.

**Execução isolada**

A função `dq_orchestrator_execute()` utiliza `dbutils.notebook.run()` para executar os notebooks filhos.

Cada chamada possui um contexto de execução próprio. Consequentemente, variáveis Python e DataFrames criados em um notebook filho não são compartilhados automaticamente com outros notebooks.

**Registro operacional**

O motor registra os resultados das execuções em um DataFrame Spark, permitindo consultas e agregações posteriores.

**Tratamento de falhas**

As exceções capturadas são registradas no log operacional.

O comportamento após uma falha depende da configuração `ORCHESTRATOR_STOP_ON_ERROR`.

---

###### Tratamento de situações especiais

Para preservar a confiabilidade operacional, o Orchestrator considera:

- **Profile inexistente:** deve ser identificado durante a validação do plano;
- **Dependência inexistente:** deve ser identificada na validação do catálogo;
- **Dependência cíclica:** impede a construção de um plano válido;
- **Falha de execução:** registrada com status `ERRO`;
- **Dependência com falha:** pode produzir status `BLOQUEADO`;
- **Interrupção antecipada:** pode resultar em Profiles planejados sem registros de execução;
- **Tempo limite:** aplicado individualmente às chamadas dos notebooks;
- **Resultado vazio:** exige tratamento adequado nas etapas de consolidação;
- **Execução parcial:** não deve ser interpretada como execução completa.

A existência de registros operacionais não garante que todos os Profiles planejados tenham sido executados.

---

###### Interpretação dos resultados

Os status operacionais representam o comportamento do motor durante a execução.

**SUCESSO**

Indica que a chamada ao notebook foi concluída sem exceção capturada pelo Orchestrator.

**ERRO**

Indica que ocorreu uma exceção durante a execução do notebook.

**BLOQUEADO**

Indica que o Profile não foi executado devido à falha de uma dependência.

**Princípio fundamental:**

> Sucesso operacional não significa aprovação da qualidade dos dados.

Um Profile pode executar corretamente e identificar problemas relevantes no dataset.

Da mesma forma, um Profile pode apresentar falha técnica sem que isso represente, por si só, um problema nos dados analisados.

A avaliação da qualidade pertence às métricas, evidências e políticas de qualidade do framework.

---

###### Características do Orchestrator

O Framework Orchestrator foi desenvolvido considerando os seguintes princípios:

- **Genérico:** não depende de regras específicas de negócio;
- **Reutilizável:** pode coordenar análises de diferentes datasets;
- **Modular:** mantém a execução separada das análises;
- **Configurável:** permite selecionar Profiles e parâmetros;
- **Rastreável:** associa registros a um identificador de execução;
- **Observável:** registra status, durações e falhas;
- **Determinístico no planejamento:** respeita as dependências configuradas;
- **Extensível:** permite adicionar novos Profiles ao catálogo;
- **Não destrutivo:** não modifica diretamente os dados analisados;
- **Independente da fonte:** coordena componentes sem exigir conhecimento do domínio dos dados.

A implementação atual utiliza execução sequencial dos notebooks, característica que deverá ser considerada nas análises de performance.

---

###### DataFrames de resultado

O Orchestrator disponibiliza estruturas padronizadas para acompanhamento operacional.

**`orchestrator_plan_df`**

Representa o planejamento dos Profiles e permite inspecionar a sequência prevista para execução.

**`orchestrator_execution_df`**

Registra os resultados operacionais dos Profiles, utilizando os campos:

- `run_id`;
- `profile`;
- `status`;
- `inicio`;
- `fim`;
- `duracao_segundos`;
- `mensagem`.

**`orchestrator_failures_df`**

Consolida os registros relacionados a falhas e bloqueios, facilitando a investigação de problemas operacionais.

**`orchestrator_metrics_df`**

Apresenta indicadores agregados por execução e status, incluindo quantidade de registros e duração total.

**`orchestrator_summary_df`**

Disponibiliza uma visão executiva da execução, incluindo quantidades por status e duração total registrada.

---

###### Limitações

A implementação atual apresenta algumas limitações que devem ser consideradas:

1. **Execução sequencial:** os notebooks são processados individualmente, sem paralelismo implementado no motor.

2. **Isolamento de contexto:** DataFrames e variáveis Python não são compartilhados automaticamente entre notebooks filhos.

3. **Possibilidade de reprocessamento:** dependências carregadas por `%run` podem provocar novas leituras e operações Spark.

4. **Persistência não garantida:** o registro operacional não assegura que os resultados analíticos dos Profiles estejam disponíveis para outros notebooks.

5. **Dependência do ambiente:** a execução depende dos recursos e das funcionalidades disponíveis no Databricks.

6. **Ausência de histórico persistente:** os resultados operacionais em memória não constituem, isoladamente, um histórico durável.

7. **Sucesso técnico limitado:** a conclusão de um notebook não comprova a validade analítica dos resultados produzidos.

Essas limitações deverão orientar a evolução do framework nas próximas etapas de estabilização, persistência e observabilidade.

---

###### Resultado esperado

Ao final da execução, o Framework Orchestrator deverá permitir responder:

1. Quais Profiles foram selecionados?
2. Quais componentes foram incluídos no plano?
3. Quais dependências foram identificadas?
4. Qual foi a ordem planejada de execução?
5. Quais Profiles foram efetivamente executados?
6. Quais apresentaram falhas?
7. Quais foram bloqueados?
8. Quanto tempo durou cada execução?
9. Qual foi a duração total registrada?
10. Os contratos operacionais foram preservados?
11. A execução foi completa ou interrompida antecipadamente?
12. Quais componentes devem ser investigados ou otimizados?

Os resultados poderão ser utilizados posteriormente na construção do histórico de execuções, nos indicadores de observabilidade e na integração do framework com práticas de **DataOps**.

**Observação:** o Framework Orchestrator deve permanecer responsável pela coordenação operacional. A interpretação das métricas e a decisão de aprovação ou reprovação dos datasets pertencem aos componentes analíticos e às políticas de qualidade.


In [0]:
%run "../bibliotecas/00_FRAMEWORK_CONFIG"

In [0]:
# ============================================================
# 11.3 CATALOGO DOS PROFILES
# ============================================================
#
# O QUE FAZ:
# Declara os notebooks e caminhos.
#
# COMO FAZ:
# Usa caminhos relativos ao notebook 11.
#
# POR QUE E IMPORTANTE:
# Evita caminhos fixos de workspace e nomes de fontes.
#
# PERGUNTA RESPONDIDA:
# "Onde cada Profile sera executado?"

ORCHESTRATOR_CATALOG = {
    "schema": {"path": "./03_SCHEMA_PROFILE", "dependencies": []},
    "distribution": {"path": "./04_DISTRIBUTION_PROFILE", "dependencies": []},
    "pattern": {"path": "./05_PATTERN_PROFILE", "dependencies": []},
    "duplicity": {"path": "./06_DUPLICITY_PROFILE", "dependencies": []},
    "consistency": {"path": "./07_CONSISTENCY_PROFILE", "dependencies": []},
    "outlier": {"path": "./08_OUTLIER_PROFILE", "dependencies": []},
    "correlation": {"path": "./09_CORRELATION_PROFILE", "dependencies": []},
    "score": {"path": "./10_DATA_QUALITY_SCORE", "dependencies": []},
}
# As dependencias internas via %run nao estao representadas neste catalogo.
# Este catalogo modela apenas dependencias de execucao entre notebooks filhos.

In [0]:
# ============================================================
# 11.4 VALIDACAO DE DEPENDENCIAS
# ============================================================
#
# O QUE FAZ:
# Valida catalogo, funcoes e selecao.
#
# COMO FAZ:
# Aplica validadores da biblioteca.
#
# POR QUE E IMPORTANTE:
# Falha cedo em configuracoes incorretas.
#
# PERGUNTA RESPONDIDA:
# "O plano de execucao e valido?"

required_helpers = (
    "dq_orchestrator_schema", "dq_orchestrator_validate_catalog",
    "dq_orchestrator_plan", "dq_orchestrator_execute"
)
missing_helpers = [name for name in required_helpers if name not in globals()]
if missing_helpers:
    raise RuntimeError("Adicione as funcoes 11 a biblioteca: " + ", ".join(missing_helpers))
if "dbutils" not in globals():
    raise RuntimeError("Este orquestrador requer Databricks com dbutils.")
dq_orchestrator_validate_catalog(ORCHESTRATOR_CATALOG)
ORCHESTRATOR_PLAN = dq_orchestrator_plan(ORCHESTRATOR_SELECTED, ORCHESTRATOR_CATALOG)
print("Plano:", " -> ".join(ORCHESTRATOR_PLAN))

In [0]:
# ============================================================
# 11.5 CONTEXTO DE EXECUCAO
# ============================================================
#
# O QUE FAZ:
# Registra identificador e plano.
#
# COMO FAZ:
# Cria DataFrame Spark com schema definido.
#
# POR QUE E IMPORTANTE:
# Garante rastreabilidade.
#
# PERGUNTA RESPONDIDA:
# "Qual e a ordem planejada?"

plan_rows = [(ORCHESTRATOR_RUN_ID, i + 1, name, ORCHESTRATOR_CATALOG[name]["path"])
             for i, name in enumerate(ORCHESTRATOR_PLAN)]
orchestrator_plan_df = spark.createDataFrame(
    plan_rows, "run_id string, ordem int, profile string, notebook_path string"
)
if ORCHESTRATOR_DISPLAY_RESULTS:
    display(orchestrator_plan_df)

In [0]:
# ============================================================
# 11.6 MOTOR DE EXECUCAO
# ============================================================
#
# O QUE FAZ:
# Executa notebooks e captura retorno, duracao e falhas.
#
# COMO FAZ:
# Usa dbutils.notebook.run em contextos isolados.
#
# POR QUE E IMPORTANTE:
# Centraliza o controle operacional.
#
# PERGUNTA RESPONDIDA:
# "Quais notebooks foram executados?"

orchestrator_execution_df, executed_plan = dq_orchestrator_execute(
    dbutils=dbutils,
    spark=spark,
    catalog=ORCHESTRATOR_CATALOG,
    selected=ORCHESTRATOR_SELECTED,
    run_id=ORCHESTRATOR_RUN_ID,
    timeout_seconds=ORCHESTRATOR_TIMEOUT_SECONDS,
    arguments=ORCHESTRATOR_NOTEBOOK_ARGUMENTS,
    stop_on_error=ORCHESTRATOR_STOP_ON_ERROR,
)
if ORCHESTRATOR_DISPLAY_RESULTS:
    display(orchestrator_execution_df)

In [0]:
# ============================================================
# 11.7 TRATAMENTO DE FALHAS
# ============================================================
#
# O QUE FAZ:
# Identifica erros e etapas ignoradas.
#
# COMO FAZ:
# Filtra o log tipado de execucao.
#
# POR QUE E IMPORTANTE:
# Facilita diagnostico sem ocultar falhas.
#
# PERGUNTA RESPONDIDA:
# "Quais Profiles falharam?"

orchestrator_failures_df = orchestrator_execution_df.where(
    F.col("status").isin("FAILED", "SKIPPED_DEPENDENCY")
)
if ORCHESTRATOR_DISPLAY_RESULTS:
    display(orchestrator_failures_df)

In [0]:

# ============================================================
# 11.8 STATUS E TEMPOS
# ============================================================
#
# O QUE FAZ:
# Calcula indicadores operacionais por status de execução.
#
# COMO FAZ:
# Agrupa os registros do Orchestrator por run_id e status,
# contabilizando as execuções e somando suas durações.
#
# POR QUE É IMPORTANTE:
# Permite acompanhar o desempenho operacional do framework,
# identificar falhas e quantificar o tempo de processamento.
#
# PERGUNTA RESPONDIDA:
# "Quantos Profiles foram executados e quanto tempo duraram?"

orchestrator_metrics_df = (
    orchestrator_execution_df
    .groupBy("run_id", "status")
    .agg(
        F.count("*").alias("quantidade"),
        F.round(
            F.sum("duracao_segundos"), 3
        ).alias("duracao_total_segundos")
    )
    .orderBy("run_id", "status")
)

if ORCHESTRATOR_DISPLAY_RESULTS:
    display(orchestrator_metrics_df)


In [0]:

# ============================================================
# 11.9 CONSOLIDACAO DO RESUMO OPERACIONAL
# ============================================================
#
# O QUE FAZ:
# Consolida os resultados da execução dos Profiles.
#
# COMO FAZ:
# Agrega as quantidades por status e soma as durações
# registradas no log de execução.
#
# POR QUE É IMPORTANTE:
# Disponibiliza uma visão geral do processamento sem
# executar novamente os notebooks.
#
# PERGUNTA RESPONDIDA:
# "Quantos Profiles foram executados, falharam ou
# foram bloqueados, e qual foi a duração total?"

orchestrator_summary_df = (
    orchestrator_execution_df
    .groupBy("run_id")
    .agg(
        F.count("*").alias("total_profiles"),
        F.sum(
            F.when(F.col("status") == "SUCESSO", 1).otherwise(0)
        ).alias("profiles_sucesso"),
        F.sum(
            F.when(F.col("status") == "ERRO", 1).otherwise(0)
        ).alias("profiles_erro"),
        F.sum(
            F.when(F.col("status") == "BLOQUEADO", 1).otherwise(0)
        ).alias("profiles_bloqueados"),
        F.round(
            F.sum("duracao_segundos"), 3
        ).alias("duracao_total_segundos")
    )
)


In [0]:

# ============================================================
# 11.10 RESUMO EXECUTIVO
# ============================================================
#
# O QUE FAZ:
# Exibe o resumo consolidado e o status operacional final.
#
# COMO FAZ:
# Consulta os DataFrames de resumo e execução,
# contabilizando falhas e execuções bloqueadas.
#
# POR QUE É IMPORTANTE:
# Facilita a identificação de problemas e permite
# verificar se o Orchestrator terminou corretamente.
#
# PERGUNTA RESPONDIDA:
# "A execução dos Profiles terminou sem falhas?"

if ORCHESTRATOR_DISPLAY_RESULTS:
    display(orchestrator_summary_df)

    display(
        orchestrator_execution_df.orderBy(
            "inicio", "profile"
        )
    )

failed_count = (
    orchestrator_execution_df
    .filter(F.col("status") == "ERRO")
    .count()
)

blocked_count = (
    orchestrator_execution_df
    .filter(F.col("status") == "BLOQUEADO")
    .count()
)

print(f"Run ID: {ORCHESTRATOR_RUN_ID}")
print(f"Profiles com erro: {failed_count}")
print(f"Profiles bloqueados: {blocked_count}")

if failed_count > 0 or blocked_count > 0:
    print("STATUS OPERACIONAL: EXECUÇÃO COM PROBLEMAS")
else:
    print("STATUS OPERACIONAL: SEM FALHAS REGISTRADAS")

# IMPORTANTE:
# SUCESSO significa que a execução do notebook terminou
# sem erro registrado pelo Orchestrator.
# Não significa que o dataset foi aprovado em qualidade.


In [0]:

# ============================================================
# 11.11 TESTES DE INTEGRACAO
# ============================================================
#
# O QUE FAZ:
# Valida os contratos de execução do Orchestrator,
# incluindo planejamento, dependências, schema,
# identificação dos Profiles e status operacionais.
#
# COMO FAZ:
# Executa validações sobre o catálogo, o plano
# e o DataFrame de execução já produzido.
#
# POR QUE É IMPORTANTE:
# Detecta inconsistências entre os componentes
# e evita regressões sem reexecutar os notebooks filhos.
#
# PERGUNTA RESPONDIDA:
# "O Orchestrator preserva seus contratos de execução?"

# ------------------------------------------------------------
# 1. VALIDACAO DO CATALOGO E DO PLANO
# ------------------------------------------------------------

assert dq_orchestrator_validate_catalog(
    ORCHESTRATOR_CATALOG
), "O catálogo do Orchestrator é inválido."

plano_esperado = dq_orchestrator_plan(
    ORCHESTRATOR_SELECTED,
    ORCHESTRATOR_CATALOG
)

assert len(plano_esperado) == len(set(plano_esperado)), (
    "O plano contém Profiles duplicados."
)

assert set(ORCHESTRATOR_SELECTED).issubset(
    set(plano_esperado)
), "Existem Profiles selecionados ausentes do plano."

assert list(ORCHESTRATOR_PLAN) == plano_esperado, (
    "O plano armazenado difere do plano calculado."
)

# ------------------------------------------------------------
# 2. VALIDACAO DA ORDEM DAS DEPENDENCIAS
# ------------------------------------------------------------

posicoes = {
    profile: indice
    for indice, profile in enumerate(plano_esperado)
}

for profile in plano_esperado:
    dependencias = ORCHESTRATOR_CATALOG[
        profile
    ].get("dependencies", [])

    for dependencia in dependencias:
        assert dependencia in posicoes, (
            f"Dependência ausente: {dependencia}"
        )

        assert posicoes[dependencia] < posicoes[profile], (
            f"Ordem inválida: {dependencia} deve "
            f"preceder {profile}."
        )

# ------------------------------------------------------------
# 3. VALIDACAO DO SCHEMA DE EXECUCAO
# ------------------------------------------------------------

schema_esperado = dq_orchestrator_schema()

assert (
    orchestrator_execution_df.schema == schema_esperado
), "O schema de execução difere do contrato esperado."

# ------------------------------------------------------------
# 4. VALIDACAO DOS REGISTROS DE EXECUCAO
# ------------------------------------------------------------

registros = orchestrator_execution_df.select(
    "run_id",
    "profile",
    "status",
    "duracao_segundos"
).collect()

profiles_registrados = [
    registro["profile"]
    for registro in registros
]

assert len(registros) <= len(plano_esperado), (
    "Há mais registros de execução do que Profiles planejados."
)

assert len(profiles_registrados) == len(
    set(profiles_registrados)
), "Existem Profiles duplicados no log."

assert set(profiles_registrados).issubset(
    set(plano_esperado)
), "O log contém Profiles fora do plano."

# ------------------------------------------------------------
# 5. VALIDACAO DOS STATUS
# ------------------------------------------------------------

status_validos = {
    "SUCESSO",
    "ERRO",
    "BLOQUEADO"
}

for registro in registros:
    assert registro["status"] in status_validos, (
        f"Status inválido: {registro['status']}"
    )

    assert registro["run_id"] == str(
        ORCHESTRATOR_RUN_ID
    ), "Foi encontrado um run_id diferente do esperado."

    duracao = registro["duracao_segundos"]

    assert duracao is not None and duracao >= 0, (
        f"Duração inválida para {registro['profile']}."
    )

# ------------------------------------------------------------
# 6. VALIDACAO DA SEQUENCIA DE EXECUCAO
# ------------------------------------------------------------

assert profiles_registrados == plano_esperado[
    :len(profiles_registrados)
], "A sequência registrada difere do plano esperado."

# ------------------------------------------------------------
# 7. RESULTADO DOS TESTES
# ------------------------------------------------------------

print("==========================================")
print("TESTES DE INTEGRACAO - ORCHESTRATOR")
print("==========================================")
print("Catálogo: OK")
print("Plano de execução: OK")
print("Ordem das dependências: OK")
print("Schema: OK")
print("Identificação dos Profiles: OK")
print("Status operacionais: OK")
print("Duração das execuções: OK")
print("------------------------------------------")
print(f"Run ID: {ORCHESTRATOR_RUN_ID}")
print(f"Profiles planejados: {len(plano_esperado)}")
print(f"Profiles registrados: {len(registros)}")
print("==========================================")
print("TESTES DE CONTRATO CONCLUIDOS.")
